# Along-Track Cost Decomposition — Compute

Per-leg decomposed costs for the best route per test case (72 routes).
Saves to `024_best_routes_per_leg_costs.parquet`.

In [1]:
from pathlib import Path

import geopandas as gpd
import msgpack
import numpy as np
import pandas as pd
from tqdm import tqdm
import warnings

from experiment_params import FORCING_SCENARIOS
from ship_routing.app.routing import RoutingResult
from ship_routing.core.config import SHIP_DEFAULT, PHYSICS_DEFAULT
from ship_routing.core.cost import power_maintain_speed_decomposed
from ship_routing.core.data import load_currents, load_waves, load_winds, select_data_for_leg
from ship_routing.core.routes import Route, WayPoint
from load_tuning_results import add_derived_features, filter_suspicious_routes

warnings.filterwarnings("ignore")

## Identify best route per test case

In [2]:
# Load decomposed costs and join with prelim metadata
files = sorted(Path("../results").glob("decomposed_costs_*.parquet"))
df_costs = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)

gdf = gpd.read_parquet("../results/results_prelim.geoparquet")
gdf = add_derived_features(gdf)
gdf = filter_suspicious_routes(gdf)
gdf = gdf[gdf.forcing_scenario_name == "baseline"].copy()
gdf = gdf.reset_index()
gdf = gdf.merge(df_costs, on=["filename", "n_elite"], how="inner")

gdf["month"] = pd.to_datetime(gdf.journey_time_start).dt.month_name().str[:3]
gdf["direction"] = gdf.journey_name.map(
    {"Atlantic_forward": "eastward", "Atlantic_backward": "westward"}
)

# Best route per test case
best_idx = gdf.groupby(
    ["month", "journey_speed_knots", "direction"]
)["elite_cost_absolute"].idxmin()
gdf_best = gdf.loc[best_idx].copy()
print(f"Best routes: {len(gdf_best)}")

# Save route keys for cross-checking with other notebooks
route_keys = gdf_best[["month", "direction", "journey_speed_knots", "filename", "n_elite",
                        "elite_cost_absolute", "cost_calm", "cost_waves", "cost_wind",
                        "cost_total", "delta_current_on_calm", "delta_current_on_waves",
                        "delta_current_total"]].copy()
route_keys.to_parquet("../results/024_best_route_keys.parquet", index=False)
print(f"Saved route keys to ../results/024_best_route_keys.parquet")

# Which msgpack files do we need?
needed_files = gdf_best.filename.unique()
print(f"Need to load from {len(needed_files)} unique filenames")

0.16% suspicious routes


Best routes: 72
Saved route keys to ../results/024_best_route_keys.parquet
Need to load from 72 unique filenames


## Load forcing data

In [3]:
bounds = gdf.total_bounds
spatial_bounds = (bounds[0] - 5, bounds[2] + 5, bounds[1] - 5, bounds[3] + 5)

baseline = FORCING_SCENARIOS["baseline"]
time_start = np.datetime64("2021-01-01")
time_end = np.datetime64("2021-12-31T23:59:59")
data_prefix = Path("..")

print("Loading currents...")
currents = load_currents(
    data_prefix / baseline["currents_path"],
    time_start=time_start, time_end=time_end,
    engine=baseline["engine"], spatial_bounds=spatial_bounds,
    load_eagerly=True,
)

print("Loading waves...")
waves = load_waves(
    data_prefix / baseline["waves_path"],
    time_start=time_start, time_end=time_end,
    engine=baseline["engine"], spatial_bounds=spatial_bounds,
    load_eagerly=True,
)

print("Loading winds...")
winds = load_winds(
    data_prefix / baseline["winds_path"],
    time_start=time_start, time_end=time_end,
    engine=baseline["engine"], spatial_bounds=spatial_bounds,
    load_eagerly=True,
)
print("Done.")

Loading currents...


Loading waves...


Loading winds...


Done.


## Compute per-leg decomposed costs

In [4]:
def route_from_routing_result(rr, elite_idx=0):
    """Extract a Route with ns-precision timestamps from a RoutingResult."""
    route = rr.elite_population.members[elite_idx].route
    fixed_waypoints = [
        WayPoint(
            lon=wp.lon, lat=wp.lat,
            time=np.datetime64(wp.time, "ns"),
        )
        for wp in route.way_points
    ]
    return Route(way_points=tuple(fixed_waypoints))


def per_leg_decomposition(route):
    """Compute per-leg decomposed costs with leg midpoint coordinates.

    Includes delta_current_on_wind which cost_through_decomposed discards.
    """
    records = []
    cum_distance = 0.0
    cum_time_s = 0.0
    for leg in route.legs:
        costs = leg.cost_through_decomposed(
            current_data_set=currents,
            wind_data_set=winds,
            wave_data_set=waves,
            ship=SHIP_DEFAULT,
            physics=PHYSICS_DEFAULT,
        )

        # Compute wind no-current effect ourselves (routes.py discards it)
        u_ship_og, v_ship_og = leg.uv_over_ground_ms
        ds_wind = select_data_for_leg(
            ds=winds,
            lon_start=leg.way_point_start.lon, lat_start=leg.way_point_start.lat,
            time_start=leg.way_point_start.time,
            lon_end=leg.way_point_end.lon, lat_end=leg.way_point_end.lat,
            time_end=leg.way_point_end.time,
        )
        ds_wave = select_data_for_leg(
            ds=waves,
            lon_start=leg.way_point_start.lon, lat_start=leg.way_point_start.lat,
            time_start=leg.way_point_start.time,
            lon_end=leg.way_point_end.lon, lat_end=leg.way_point_end.lat,
            time_end=leg.way_point_end.time,
        )
        _, _, pwr_wind_nc = power_maintain_speed_decomposed(
            u_current_ms=0, v_current_ms=0,
            u_wind_ms=ds_wind.uw, v_wind_ms=ds_wind.vw,
            w_wave_height=ds_wave.wh,
            u_ship_og_ms=u_ship_og, v_ship_og_ms=v_ship_og,
            ship=SHIP_DEFAULT, physics=PHYSICS_DEFAULT,
        )
        dt = leg.duration_seconds
        cost_wind_nc = pwr_wind_nc.mean().data[()] * dt

        costs["cost_wind_no_current"] = cost_wind_nc
        costs["delta_current_on_calm"] = costs["cost_calm_no_current"] - costs["cost_calm"]
        costs["delta_current_on_waves"] = costs["cost_waves_no_current"] - costs["cost_waves"]
        costs["delta_current_on_wind"] = cost_wind_nc - costs["cost_wind"]
        costs["delta_current_total"] = (
            costs["delta_current_on_calm"]
            + costs["delta_current_on_waves"]
            + costs["delta_current_on_wind"]
        )

        mid_lon = (leg.way_point_start.lon + leg.way_point_end.lon) / 2
        mid_lat = (leg.way_point_start.lat + leg.way_point_end.lat) / 2
        leg_len = leg.length_meters
        cum_distance += leg_len
        dt_s = (leg.way_point_end.time - leg.way_point_start.time) / np.timedelta64(1, "s")
        cum_time_s += dt_s

        costs["mid_lon"] = mid_lon
        costs["mid_lat"] = mid_lat
        costs["leg_length_m"] = leg_len
        costs["cum_distance_km"] = cum_distance / 1e3
        costs["duration_s"] = dt_s
        costs["cum_time_h"] = (cum_time_s - dt_s / 2) / 3600  # midpoint time
        records.append(costs)
    return pd.DataFrame(records)

In [5]:
# Build lookup: filename -> (msgpack_file, key_in_file)
# The filename in the prelim table is the key inside the msgpack
msgpack_files = sorted(Path("../results/").glob("results_ablation_baseline_*.msgpack"))
msgpack_files = [f for f in msgpack_files if "crosseval" not in f.name]

# Collect per-leg data for all best routes
all_leg_data = []

for mf in tqdm(msgpack_files, desc="msgpack files"):
    with open(mf, "rb") as f:
        raw = msgpack.unpack(f, raw=False)

    # Check which best routes come from this file
    keys_in_file = set(raw.keys()) & set(needed_files)
    if not keys_in_file:
        del raw
        continue

    best_from_file = gdf_best[gdf_best.filename.isin(keys_in_file)]

    for _, row in best_from_file.iterrows():
        key = row.filename
        elite_idx = int(row.n_elite)
        if key not in raw:
            continue

        rr = RoutingResult.from_msgpack(raw[key])
        route = route_from_routing_result(rr, elite_idx)
        df_legs = per_leg_decomposition(route)
        df_legs["month"] = row.month
        df_legs["direction"] = row.direction
        df_legs["speed"] = row.journey_speed_knots
        df_legs["filename"] = key
        all_leg_data.append(df_legs)

    del raw

df_all_legs = pd.concat(all_leg_data, ignore_index=True)
print(f"Total legs: {len(df_all_legs)} across {len(all_leg_data)} routes")

msgpack files:   0%|          | 0/13 [00:00<?, ?it/s]

msgpack files:   8%|▊         | 1/13 [00:12<02:29, 12.44s/it]

msgpack files:  15%|█▌        | 2/13 [00:24<02:14, 12.24s/it]

msgpack files:  23%|██▎       | 3/13 [00:53<03:18, 19.89s/it]

msgpack files:  31%|███       | 4/13 [01:03<02:25, 16.12s/it]

...

msgpack files:  85%|████████▍ | 11/13 [02:02<00:21, 10.52s/it]

msgpack files:  92%|█████████▏| 12/13 [02:27<00:14, 14.98s/it]

msgpack files: 100%|██████████| 13/13 [02:31<00:00, 11.53s/it]

msgpack files: 100%|██████████| 13/13 [02:31<00:00, 11.62s/it]

Total legs: 4269 across 72 routes


## Save per-leg data

In [6]:
df_all_legs.to_parquet("../results/024_best_routes_per_leg_costs.parquet", index=False)
print(f"Saved {len(df_all_legs)} leg records to ../results/024_best_routes_per_leg_costs.parquet")

Saved 4269 leg records to ../results/024_best_routes_per_leg_costs.parquet
